# E1b UNet Normal-Aware on Preprocessed BTXRD 224x224

Kaggle notebook for training the normal-aware image-only UNet baseline on GPU T4. Attach a Kaggle Dataset that contains the exported BTXRD structure:

```text
data/exports/btxrd_preprocessed/train.csv
data/exports/btxrd_preprocessed/val.csv
data/exports/btxrd_preprocessed/test.csv
data/processed/images_preprocessed/
data/processed/masks_preprocessed/
```

If your Kaggle Dataset root is different, edit only `DATA_ROOT` in the setup cell.

## Clone Repo

Run this cell first after starting or restarting the Kaggle session. It downloads the exact branch that contains the E1b normal-aware UNet pipeline.


In [ ]:
!rm -rf /kaggle/working/BTXRD-LViT
!git clone -b model/e1-unet-baseline https://github.com/lehngoc/BTXRD-LViT.git /kaggle/working/BTXRD-LViT


In [ ]:
import os
import sys
import json
import shutil
import zipfile
from pathlib import Path

import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Setup Paths

The default `DATA_ROOT` below matches the Kaggle input path already confirmed for this run. If you create a new Kaggle Dataset version with a different path, edit only `DATA_ROOT`.


In [ ]:
# Default paths for the current Kaggle run. Edit DATA_ROOT only if your input path changes.
REPO_ROOT = Path("/kaggle/working/BTXRD-LViT")
DATA_ROOT = Path("/kaggle/input/datasets/lehngoc/btxrd-preprocessed-dataset/btxrd-preprocessed")

def find_data_root():
    target = Path("data/exports/btxrd_preprocessed/train.csv")
    if DATA_ROOT.exists() and (DATA_ROOT / target).exists():
        return DATA_ROOT

    for root in Path("/kaggle/input").glob("**"):
        if root.is_dir() and (root / target).exists():
            return root

    raise FileNotFoundError("Could not find BTXRD train.csv under /kaggle/input. Update DATA_ROOT.")

DATA_ROOT = find_data_root()
OUTPUT_DIR = Path("/kaggle/working/experiments/E1_unet_preprocessed_224_weighted_loss")
RUNTIME_CONFIG = Path("/kaggle/working/e1_unet_kaggle.yaml")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT:", REPO_ROOT)
print("REPO_ROOT exists:", REPO_ROOT.exists())
print("DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
required_paths = [
    DATA_ROOT / 'data/exports/btxrd_preprocessed/train.csv',
    DATA_ROOT / 'data/exports/btxrd_preprocessed/val.csv',
    DATA_ROOT / 'data/exports/btxrd_preprocessed/test.csv',
    DATA_ROOT / 'data/processed/images_preprocessed',
    DATA_ROOT / 'data/processed/masks_preprocessed',
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required BTXRD paths:\n' + '\n'.join(missing))

print('BTXRD dataset paths are ready.')

## Runtime Config

The source config stays unchanged. This cell writes a Kaggle-specific copy with `root_dir`, `num_workers`, and output path adjusted for `/kaggle/working`.

In [ ]:
import yaml

BASE_CONFIG = REPO_ROOT / 'configs/train_unet_baseline.yaml'
with BASE_CONFIG.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['data']['root_dir'] = str(DATA_ROOT)
cfg['training']['device'] = 'cuda'
cfg['training']['image_size'] = 224
cfg['training']['batch_size'] = 4
cfg['training']['num_workers'] = 2
cfg['training']['epochs'] = 200
cfg['training']['learning_rate'] = 0.0003
cfg['training']['positive_weight'] = 20.0
cfg['training']['dice_on_tumor_only'] = True
cfg['training']['early_stopping_patience'] = 100
cfg['training']['output_dir'] = str(OUTPUT_DIR)

with RUNTIME_CONFIG.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(RUNTIME_CONFIG.read_text())

## Smoke Test

Run this before the long training job. It checks dataset loading, binary masks, UNet forward pass, and backward pass.

In [ ]:
!python {REPO_ROOT}/src/training/smoke_test_model_pipeline.py --config {RUNTIME_CONFIG} --samples-per-split 8

## Train E1 UNet Baseline

This is the main run. On Kaggle T4, keep GPU enabled: Notebook Settings -> Accelerator -> GPU T4.

In [ ]:
!python {REPO_ROOT}/src/training/train_unet.py --config {RUNTIME_CONFIG} --device cuda

## Evaluate Best Checkpoint

Metrics are written separately for validation and test. For normal cases, prioritize `normal_pred_area_ratio` and `normal_fp_image_rate` in the report.

In [ ]:
BEST_CKPT = OUTPUT_DIR / 'best.pt'
assert BEST_CKPT.exists(), f'Missing checkpoint: {BEST_CKPT}'

!python {REPO_ROOT}/src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {BEST_CKPT} --split val --device cuda --output {OUTPUT_DIR}/val_metrics.json
!python {REPO_ROOT}/src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {BEST_CKPT} --split test --device cuda --output {OUTPUT_DIR}/test_metrics.json

In [ ]:
def load_metrics(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)

for split in ['val', 'test']:
    metrics = load_metrics(OUTPUT_DIR / f'{split}_metrics.json')
    print(f'[{split}]')
    for key in ['tumor_dice', 'tumor_iou', 'normal_pred_area_ratio', 'normal_fp_image_rate']:
        print(f'  {key}: {metrics[key]:.6f}')

## Threshold Sweep

Use this to inspect the tumor Dice versus normal false-positive trade-off for the best checkpoint.


In [ ]:
sweep_rows = []
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    out_path = OUTPUT_DIR / f"val_metrics_thr{int(threshold * 100):02d}.json"
    !python {REPO_ROOT}/src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {BEST_CKPT} --split val --device cuda --threshold {threshold} --output {out_path}
    metrics = load_metrics(out_path)
    sweep_rows.append({
        "threshold": threshold,
        "tumor_dice": metrics["tumor_dice"],
        "tumor_iou": metrics["tumor_iou"],
        "normal_pred_area_ratio": metrics["normal_pred_area_ratio"],
        "normal_fp_image_rate": metrics["normal_fp_image_rate"],
    })

sweep_path = OUTPUT_DIR / "threshold_sweep_metrics.json"
with sweep_path.open("w", encoding="utf-8") as f:
    json.dump(sweep_rows, f, indent=2)

for row in sweep_rows:
    print(row)


## Package Artifacts

The zip file appears under `/kaggle/working` and can be downloaded from the Kaggle output panel.

In [ ]:
ARTIFACT_ZIP = Path('/kaggle/working/E1_unet_preprocessed_224_weighted_loss_artifacts.zip')
artifact_names = [
    'best.pt',
    'last.pt',
    'history.csv',
    'best_summary.json',
    'val_metrics.json',
    'test_metrics.json',
    'threshold_sweep_metrics.json',
    'val_metrics_thr30.json',
    'val_metrics_thr40.json',
    'val_metrics_thr50.json',
    'val_metrics_thr60.json',
    'val_metrics_thr70.json',
    'config.json',
]

with zipfile.ZipFile(ARTIFACT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in artifact_names:
        path = OUTPUT_DIR / name
        if path.exists():
            zf.write(path, arcname=name)
        else:
            print('Skipping missing artifact:', path)

print('Wrote:', ARTIFACT_ZIP)
print('Size MB:', ARTIFACT_ZIP.stat().st_size / (1024 * 1024))